In [ ]:
#Initial data cleaning for the data set. This script will remove any rows with missing values.

In [1]:
import pandas as pd
import numpy as np


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


In [6]:
df = pd.read_csv("../data/raw/credit_card_transactions.csv")

In [7]:
#Inspecting data
df.head()
df.info()
df.describe()
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 24 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  str    
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  str    
 4   category               1296675 non-null  str    
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  str    
 7   last                   1296675 non-null  str    
 8   gender                 1296675 non-null  str    
 9   street                 1296675 non-null  str    
 10  city                   1296675 non-null  str    
 11  state                  1296675 non-null  str    
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long                   129667

(1296675, 24)

In [8]:
#Check for missing values
df.isnull().sum()

Unnamed: 0                    0
trans_date_trans_time         0
cc_num                        0
merchant                      0
category                      0
amt                           0
first                         0
last                          0
gender                        0
street                        0
city                          0
state                         0
zip                           0
lat                           0
long                          0
city_pop                      0
job                           0
dob                           0
trans_num                     0
unix_time                     0
merch_lat                     0
merch_long                    0
is_fraud                      0
merch_zipcode            195973
dtype: int64

In [9]:
#We see that merch_zipcode has the most missing values so we will drop that column
df = df.drop(columns=["merch_zipcode"])

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
#Converting date columns
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df["dob"] = pd.to_datetime(df["dob"])

In [13]:
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df["dob"] = pd.to_datetime(df["dob"])

In [14]:
#beginning feature engineering
#transaction hour
df["hour"] = df["trans_date_trans_time"].dt.hour

In [15]:
#day of the week
df["day_of_week"] = df["trans_date_trans_time"].dt.day_name()

In [16]:
#create column for weekend indicator, as most transactions take place on weekdays
df["is_weekend"] = df["trans_date_trans_time"].dt.dayofweek >= 5

In [18]:
#distance between customer and merchant
from geopy.distance import geodesic

df["distance_km"] = df.apply(
    lambda row: geodesic(
        (row["lat"], row["long"]),
        (row["merch_lat"], row["merch_long"])
    ).km,
    axis=1
)

In [19]:
#transaction amount categories
df["amount_category"] = pd.cut(
    df["amt"],
    bins=[0,20,100,500,5000],
    labels=["Very Small","Small","Medium","Large"]
)

In [20]:
#Save cleaned data
df.to_csv(
    "../data/processed/cleaned_transactions.csv",
    index=False
)